In [9]:
import os
import librosa
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

X = [] 
y = [] 

def get_mfcc(file_path):
    try:
        # 1. Load audio file
        audio, sr = librosa.load(file_path, sr=16000)
        
        # 2. FEATURE ENGINEERING: Trim silence from beginning and end
        # top_db=25 means any sound 25dB below the peak volume is considered silence and removed
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        # Filter out extremely short noises (e.g., clicks shorter than 0.15 seconds)
        if len(trimmed_audio) < 2400: 
            return None

        # 3. Extract MFCC on the PURE spoken word
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T # Transpose to shape (frames, 13)

        # 4. FEATURE ENGINEERING: Dynamic Temporal Pooling
        # Dynamically split the frames into 3 equal sections, regardless of length
        parts = np.array_split(mfccs, 3)
        
        # Calculate the mean for each dynamic part
        part1 = np.mean(parts[0], axis=0) # Always the start of the word
        part2 = np.mean(parts[1], axis=0) # Always the middle
        part3 = np.mean(parts[2], axis=0) # Always the end

        # Concatenate them into a single 1D array of 39 features
        features = np.concatenate((part1, part2, part3))
        
        return features
    except Exception as e:
        return None

print("Processing Target Words (with Silence Trimming)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        print(f"Warning: Directory for '{word}' not found.")
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(label_idx)

print("\nProcessing Unknown Words (with Silence Trimming)...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir):
        continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word} (Unknown)"): 
        file_path = os.path.join(word_dir, file_name)
        mfcc_data = get_mfcc(file_path)
        if mfcc_data is not None:
            X.append(mfcc_data)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nFeature extraction complete! Saved {len(X)} robust samples.")
print(f"NEW Features shape: {X.shape} (Dynamic Temporal Pooling)")

Processing Target Words (with Silence Trimming)...


Loading on:   0%|          | 0/1500 [00:00<?, ?it/s]

Loading stop: 100%|██████████| 1500/1500 [00:16<00:00, 93.28it/s] 



Processing Unknown Words (with Silence Trimming)...


Loading down (Unknown): 100%|██████████| 375/375 [00:04<00:00, 92.87it/s]


Feature extraction complete! Saved 5995 robust samples.
NEW Features shape: (5995, 39) (Dynamic Temporal Pooling)


In [12]:
import os
import wave
import numpy as np
from tqdm import tqdm

DATA_PATH = '../Data/speech_commands/'
SAVE_PATH = '../Data/'

TARGET_WORDS = ['on', 'off', 'stop']
UNKNOWN_WORDS = ['right', 'left', 'up', 'down']

# ==========================================
# 1. EXACT MICROPYTHON FUNCTIONS (COPIED FROM BOARD)
# ==========================================
def calculate_energy(buffer, start=0, end=None):
    if end is None:
        end = len(buffer)
    sum_squares = 0.0
    # Process exactly as 16-bit PCM bytes
    for i in range(start, end, 2):
        val = buffer[i] | (buffer[i+1] << 8)
        if val >= 32768: val -= 65536
        sum_squares += val * val
    num_samples = (end - start) // 2
    if num_samples == 0: return 0
    return (sum_squares / num_samples) ** 0.5

def get_trim_indices(raw_buffer, noise_floor):
    window_size = 512 
    buffer_len = len(raw_buffer)
    start_idx = 0
    end_idx = buffer_len
    
    for i in range(0, buffer_len, window_size):
        chunk_end = min(i + window_size, buffer_len)
        if calculate_energy(raw_buffer, i, chunk_end) > noise_floor * 1.5:
            start_idx = max(0, i - (window_size * 2))
            break
            
    for i in range(buffer_len - window_size, -1, -window_size):
        chunk_end = min(i + window_size, buffer_len)
        if calculate_energy(raw_buffer, i, chunk_end) > noise_floor * 1.5:
            end_idx = min(buffer_len, chunk_end + (window_size * 2))
            break
            
    if start_idx >= end_idx - 1000:
        return 0, buffer_len
    return start_idx, end_idx

# ==========================================
# SUPER-ROBUST TIME-DOMAIN FEATURE EXTRACTOR
# ==========================================
def compute_smart_features(raw_buffer, start_byte, end_byte, num_coeffs=13):
    num_samples = (end_byte - start_byte) // 2
    if num_samples <= 0: return [0.0] * num_coeffs
    
    max_amp = 1
    sum_amp = 0
    sum_high_freq = 0
    sum_low_freq = 0
    zcr = 0
    peaks = 0
    
    last_val = 0
    last_low = 0
    last_sign = 0
    last_diff_sign = 0
    
    envelope = [0.0] * 8
    samples_per_bin = max(1, num_samples // 8)
    
    sample_idx = 0
    
    for i in range(start_byte, end_byte, 2):
        val = raw_buffer[i] | (raw_buffer[i+1] << 8)
        if val >= 32768: val -= 65536
        abs_val = abs(val)
        
        # 1. Volume & Energy
        if abs_val > max_amp: max_amp = abs_val
        sum_amp += abs_val
        
        # 2. DIGITAL FILTERS (Poor-man's MFCC)
        # High-Pass Filter (Emphasizes 'S', 'F', 'T' sounds)
        diff = val - last_val
        sum_high_freq += abs(diff)
        
        # Low-Pass Filter (Emphasizes 'O', 'U', 'A' sounds)
        low_val = (last_low * 8 + val * 2) // 10
        sum_low_freq += abs(low_val)
        
        # 3. Envelope
        bin_idx = min(7, sample_idx // samples_per_bin)
        envelope[bin_idx] += abs_val
        
        # 4. Friction & Pitch (Noise Gated)
        if abs_val > 300: # Fixed noise gate
            sign = 1 if val > 0 else -1
            if sign != last_sign and last_sign != 0:
                zcr += 1
            last_sign = sign
            
            diff_sign = 1 if diff > 0 else -1 if diff < 0 else 0
            if diff_sign != last_diff_sign and last_diff_sign == 1:
                peaks += 1
            last_diff_sign = diff_sign
            
        last_val = val
        last_low = low_val
        sample_idx += 1
        
    # --- NORMALIZE FEATURES (0.0 to 1.0) ---
    f1 = (sum_amp / num_samples) / max_amp              
    f2 = sum_high_freq / (sum_amp + 1) # High Freq Ratio
    f3 = sum_low_freq / (sum_amp + 1)  # Low Freq Ratio
    f4 = zcr / num_samples             
    f5 = peaks / num_samples           
    
    max_env = max(envelope)
    if max_env == 0: max_env = 1
    env_norm = [e / max_env for e in envelope] 
    
    return [f1, f2, f3, f4, f5] + env_norm

def extract_39_features(raw_buffer, noise_floor):
    start_idx, end_idx = get_trim_indices(raw_buffer, noise_floor)
    trimmed_len = end_idx - start_idx
    
    bytes_per_part = ((trimmed_len // 2) // 3) * 2
    
    p1_s = start_idx
    p1_e = p1_s + bytes_per_part
    p2_s = p1_e
    p2_e = p2_s + bytes_per_part
    p3_s = p2_e
    p3_e = end_idx 
    
    # Apply the new Smart Extractor to Start, Middle, and End of the word
    m1 = compute_smart_features(raw_buffer, p1_s, p1_e, 13)
    m2 = compute_smart_features(raw_buffer, p2_s, p2_e, 13)
    m3 = compute_smart_features(raw_buffer, p3_s, p3_e, 13)
    
    return m1 + m2 + m3

# ==========================================
# 2. DATASET PROCESSING
# ==========================================
def load_wav_as_bytes(filepath):
    try:
        with wave.open(filepath, 'rb') as w:
            if w.getframerate() != 16000 or w.getsampwidth() != 2:
                return None
            frames = w.readframes(w.getnframes())
            return bytearray(frames)
    except Exception:
        return None

X = []
y = []

print("Processing Target Words (with Pyboard Emulator)...")
for label_idx, word in enumerate(TARGET_WORDS):
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir): continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:1500], desc=f"Loading {word}"):
        raw_bytes = load_wav_as_bytes(os.path.join(word_dir, file_name))
        if raw_bytes and len(raw_bytes) > 4000:
            # We estimate Kaggle studio noise floor to be around 100
            features = extract_39_features(raw_bytes, noise_floor=100.0)
            X.append(features)
            y.append(label_idx)

print("\nProcessing Unknown Words...")
unknown_label_idx = len(TARGET_WORDS) 
for word in UNKNOWN_WORDS:
    word_dir = os.path.join(DATA_PATH, word)
    if not os.path.exists(word_dir): continue
        
    files = [f for f in os.listdir(word_dir) if f.endswith('.wav')]
    for file_name in tqdm(files[:375], desc=f"Loading {word}"):
        raw_bytes = load_wav_as_bytes(os.path.join(word_dir, file_name))
        if raw_bytes and len(raw_bytes) > 4000:
            features = extract_39_features(raw_bytes, noise_floor=100.0)
            X.append(features)
            y.append(unknown_label_idx)

X = np.array(X)
y = np.array(y)

np.save(os.path.join(SAVE_PATH, 'X_features.npy'), X)
np.save(os.path.join(SAVE_PATH, 'y_labels.npy'), y)

print(f"\nAwesome! Saved {len(X)} samples using Pyboard-native algorithms.")
print(f"X shape: {X.shape}")

Processing Target Words (with Pyboard Emulator)...


Loading stop: 100%|██████████| 1500/1500 [00:14<00:00, 101.54it/s]



Processing Unknown Words...


Loading down: 100%|██████████| 375/375 [00:04<00:00, 82.52it/s]



Awesome! Saved 6000 samples using Pyboard-native algorithms.
X shape: (6000, 39)


In [19]:
import numpy as np
import os
import struct

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import optuna

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load Data
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. NEW: Standard Scaler (No data loss!)
print("\n--- Applying StandardScaler (No data compression!) ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. OPTUNA Optimization
print("\n--- Starting Optuna optimization for Logistic Regression ---")
def objective(trial):
    c_val = trial.suggest_float('C', 0.01, 100.0, log=True)
    lr = LogisticRegression(C=c_val, max_iter=5000, random_state=42)
    lr.fit(X_train_scaled, y_train)
    preds = lr.predict(X_test_scaled)
    return accuracy_score(y_test, preds)

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 4. Train Final Model
print("\n--- Training Final Lightweight Classifier ---")
final_lr = LogisticRegression(**study.best_params, max_iter=5000, random_state=42)
final_lr.fit(X_train_scaled, y_train)
preds = final_lr.predict(X_test_scaled)

target_names = ['on', 'off', 'stop', 'unknown']
print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. EXPORT NEURAL NETWORK AS BINARY (ZERO-RAM PARSING)
print("\n--- Exporting Neural Network as Binary ---")
output_file = os.path.join(OUTPUT_PATH, 'model_weights.bin')

with open(output_file, 'wb') as f:
    # 1. نوشتن ابعاد شبکه (تعداد ویژگی‌ها، نورون‌ها و کلاس‌ها)
    f.write(struct.pack('<HHH', 39, study.best_params['hidden_size'], 4))
    
    # 2. نوشتن پارامترهای نرمال‌ساز
    for val in scaler.mean_: f.write(struct.pack('<f', float(val)))
    for val in scaler.scale_: f.write(struct.pack('<f', float(val)))
    
    # 3. نوشتن لایه اول (W1, B1)
    for row in final_mlp.coefs_[0]:
        for val in row: f.write(struct.pack('<f', float(val)))
    for val in final_mlp.intercepts_[0]:
        f.write(struct.pack('<f', float(val)))
        
    # 4. نوشتن لایه خروجی (W2, B2)
    for row in final_mlp.coefs_[1]:
        for val in row: f.write(struct.pack('<f', float(val)))
    for val in final_mlp.intercepts_[1]:
        f.write(struct.pack('<f', float(val)))

print(f"Neural Network successfully exported to: {output_file}")
print("!!! ACTION REQUIRED: Move 'model_weights.bin' to your SD Card !!!")

Loaded Data -> X shape: (6000, 39), y shape: (6000,)

--- Applying StandardScaler (No data compression!) ---

--- Starting Optuna optimization for Logistic Regression ---
Best Accuracy during CV: 62.58%

--- Training Final Lightweight Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.60      0.77      0.68       300
         off       0.58      0.53      0.55       300
        stop       0.79      0.75      0.77       300
     unknown       0.53      0.46      0.49       300

    accuracy                           0.63      1200
   macro avg       0.63      0.63      0.62      1200
weighted avg       0.63      0.63      0.62      1200


--- Exporting Neural Network as Binary ---


KeyError: 'hidden_size'

In [15]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

DATA_PATH = '../Data/'
OUTPUT_PATH = '../edge_mcu'

# 1. Load the FULL 39-feature dataset
X = np.load(os.path.join(DATA_PATH, 'X_features.npy'))
y = np.load(os.path.join(DATA_PATH, 'y_labels.npy'))

print(f"Loaded Data -> X shape: {X.shape}, y shape: {y.shape}")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. STAGE 1: Dimensionality Reduction using LDA
print("\n--- Running LDA to reduce 39 features to 3 Super-Features ---")
# n_components is always (number of classes - 1) for LDA
lda = LinearDiscriminantAnalysis(n_components=3)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

print(f"New Training Data Shape after LDA: {X_train_lda.shape}")

# 3. STAGE 2: Optuna Optimization for Logistic Regression
print("\n--- Starting Optuna optimization for Logistic Regression ---")
def objective(trial):
    # Tune the regularization parameter C
    c_val = trial.suggest_float('C', 1e-4, 1e2, log=True)
    
    lr = LogisticRegression(C=c_val, max_iter=50000, random_state=42)
    lr.fit(X_train_lda, y_train)
    preds = lr.predict(X_test_lda)
    
    return accuracy_score(y_test, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print(f"\nBest Optuna Parameters: {study.best_params}")
print(f"Best Accuracy during CV: {study.best_value * 100:.2f}%")

# 4. TRAIN FINAL CLASSIFIER
print("\n--- Training Final Lightweight Classifier ---")
final_lr = LogisticRegression(**study.best_params, max_iter=50000, random_state=42)
final_lr.fit(X_train_lda, y_train)

preds = final_lr.predict(X_test_lda)
unique_classes = np.unique(y_test)
target_names = ['on', 'off', 'stop', 'unknown'][:len(unique_classes)]

print("\nFinal Classification Report:")
print(classification_report(y_test, preds, target_names=target_names))

# 5. EXPORT PURE PYTHON ARRAYS FOR EDGE INFERENCE
print("\n--- Exporting LDA and Classifier Parameters to model_data_lr.py ---")

# Extract LDA parameters
lda_xbar = lda.xbar_.tolist()         # Shape: (39,) -> The mean of each feature
lda_scalings = lda.scalings_.tolist() # Shape: (39, 3) -> The transformation matrix

# Extract Logistic Regression parameters
lr_coef = final_lr.coef_.tolist()           # Shape: (4, 3) -> Weights for the 3 super-features
lr_intercept = final_lr.intercept_.tolist() # Shape: (4,) -> Bias for each class

output_file = os.path.join(OUTPUT_PATH, 'model_data_lr.py')
with open(output_file, 'w') as f:
    f.write("# Auto-Generated LDA + Logistic Regression Export\n\n")
    f.write(f"CLASSES = {target_names}\n\n")
    
    # Write LDA arrays
    f.write("LDA_XBAR = [\n")
    f.write(f"    {[round(float(val), 6) for val in lda_xbar]}\n")
    f.write("]\n\n")
    
    f.write("LDA_SCALINGS = [\n")
    for row in lda_scalings:
        f.write(f"    {[round(float(val), 6) for val in row]},\n")
    f.write("]\n\n")
    
    # Write LR arrays
    f.write("LR_COEF = [\n")
    for class_weights in lr_coef:
        f.write(f"    {[round(float(w), 6) for w in class_weights]},\n")
    f.write("]\n\n")
    
    f.write("LR_INTERCEPT = [\n")
    f.write(f"    {[round(float(i), 6) for i in lr_intercept]}\n")
    f.write("]\n")

print(f"Pipeline successfully exported to: {output_file}")

Loaded Data -> X shape: (6000, 39), y shape: (6000,)

--- Running LDA to reduce 39 features to 3 Super-Features ---
New Training Data Shape after LDA: (4800, 3)

--- Starting Optuna optimization for Logistic Regression ---

Best Optuna Parameters: {'C': 16.98926876082803}
Best Accuracy during CV: 61.92%

--- Training Final Lightweight Classifier ---

Final Classification Report:
              precision    recall  f1-score   support

          on       0.61      0.73      0.66       300
         off       0.58      0.53      0.55       300
        stop       0.79      0.73      0.76       300
     unknown       0.51      0.49      0.50       300

    accuracy                           0.62      1200
   macro avg       0.62      0.62      0.62      1200
weighted avg       0.62      0.62      0.62      1200


--- Exporting LDA and Classifier Parameters to model_data_lr.py ---
Pipeline successfully exported to: ../edge_mcu\model_data_lr.py


In [25]:
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))

import model_data_lr

def predict_audio_class(mfcc_features):
    """
    Optimized Inference Engine with Dynamic Dimension Detection.
    """
    # Dynamically detect dimensions from the imported model
    num_original_features = len(model_data_lr.LDA_SCALINGS)      # Should be 39
    num_super_features = len(model_data_lr.LDA_SCALINGS[0])      # Should be 3 (or 2)
    num_classes = len(model_data_lr.CLASSES)
    
    # --- STAGE 1: LDA Transformation ---
    lda_features = [0.0] * num_super_features
    for i in range(num_super_features):
        sum_val = 0.0
        for j in range(num_original_features):
            sum_val += mfcc_features[j] * model_data_lr.LDA_SCALINGS[j][i]
        lda_features[i] = sum_val
        
    # --- STAGE 2: Logistic Regression Inference ---
    class_scores = [0.0] * num_classes
    for c in range(num_classes):
        # Accessing intercept correctly based on your export script
        # If model_data_lr.LR_INTERCEPT is a 1D list, use [c]
        # If it is a 2D list (as in your export code), use [0][c]
        score = model_data_lr.LR_INTERCEPT[0][c] if isinstance(model_data_lr.LR_INTERCEPT[0], list) else model_data_lr.LR_INTERCEPT[c]
        
        for i in range(num_super_features):
            score += lda_features[i] * model_data_lr.LR_COEF[c][i]
            
        class_scores[c] = score
            
    # Find the index of the highest score (Argmax)
    return class_scores.index(max(class_scores))

# --- Example Usage on Board ---
# features = [1.2, -0.5, 3.4, ...] # The 39 MFCC features extracted from the mic
# result_idx = predict_audio_class(features)
# print("Predicted Word:", model_data.CLASSES[result_idx])

import random
for j in range(10):
    i = random.randint(0, len(y_test))
    features = X_test[i]
    result_idx = predict_audio_class(features)
    print("Predicted Word:", model_data_lr.CLASSES[result_idx], "and actuall is:", y_test[i] + 1)

Predicted Word: unknown and actuall is: 3
Predicted Word: unknown and actuall is: 4
Predicted Word: unknown and actuall is: 4
Predicted Word: unknown and actuall is: 4
Predicted Word: unknown and actuall is: 3
Predicted Word: unknown and actuall is: 2
Predicted Word: unknown and actuall is: 1
Predicted Word: unknown and actuall is: 3
Predicted Word: unknown and actuall is: 2
Predicted Word: unknown and actuall is: 2


**Real test**

In [6]:
import librosa
import numpy as np
import sys
import os

sys.path.append(os.path.abspath('../edge_mcu'))


# Import your trained Kaggle model parameters
try:
    import model_data_lr
except ImportError:
    print("[!] ERROR: 'model_data_lr.py' not found in the current directory.")
    sys.exit(1)

# ==========================================
# 1. KAGGLE-STYLE FEATURE EXTRACTION
# ==========================================
def extract_features_from_wav(file_path):
    print(f"--- Processing: {file_path} ---")
    try:
        # Load audio at 16kHz to match training data
        audio, sr = librosa.load(file_path, sr=16000)
        
        # Trim silence (top_db=25 just like the Kaggle training notebook)
        trimmed_audio, _ = librosa.effects.trim(audio, top_db=25)
        
        if len(trimmed_audio) < 2400: 
            print("[!] Audio is too short or empty after trimming silence.")
            return None

        # Extract MFCC
        mfccs = librosa.feature.mfcc(y=trimmed_audio, sr=sr, n_mfcc=13, hop_length=512)
        mfccs = mfccs.T 

        # Dynamic Temporal Pooling (Split into 3 parts)
        parts = np.array_split(mfccs, 3)
        part1 = np.mean(parts[0], axis=0) 
        part2 = np.mean(parts[1], axis=0) 
        part3 = np.mean(parts[2], axis=0) 

        # Combine into 39 features
        features = np.concatenate((part1, part2, part3))
        return features

    except Exception as e:
        print(f"[!] Error processing audio: {e}")
        return None

# ==========================================
# 2. INFERENCE PIPELINE (LDA + LR)
# ==========================================
def predict(features):
    # Convert imported lists to flat numpy arrays for easy math
    # Flatten is used to safely handle both 1D [1,2] and 2D [[1,2]] exports
    xbar = np.array(model_data_lr.LDA_XBAR).flatten()
    scalings = np.array(model_data_lr.LDA_SCALINGS)
    coef = np.array(model_data_lr.LR_COEF)
    intercept = np.array(model_data_lr.LR_INTERCEPT).flatten()
    
    # STAGE 1: LDA Transformation (Mean subtraction & Matrix Multiplication)
    adjusted_features = features - xbar
    lda_features = np.dot(adjusted_features, scalings)
    
    # STAGE 2: Logistic Regression Inference
    # Dot product of super-features and weights, plus the intercept/bias
    class_scores = np.dot(lda_features, coef.T) + intercept
    
    # Find the index of the highest score
    best_idx = np.argmax(class_scores)
    
    return model_data_lr.CLASSES[best_idx], class_scores

# ==========================================
# 3. RUN TEST
# ==========================================
if __name__ == '__main__':
    # Change this to the name of your recorded WAV file
    TARGET_WAV_FILE = "../Data/r_stop.wav" 
    
    extracted_features = extract_features_from_wav(TARGET_WAV_FILE)
    
    if extracted_features is not None:
        predicted_word, scores = predict(extracted_features)
        print("\n" + "="*40)
        print(f">>> PREDICTED COMMAND: {predicted_word.upper()} <<<")
        print("="*40)
        
        print("\nConfidence Scores (Raw Logits):")
        for i, word in enumerate(model_data_lr.CLASSES):
            print(f" - {word}: {scores[i]:.4f}")

--- Processing: ../Data/r_stop.wav ---

>>> PREDICTED COMMAND: STOP <<<

Confidence Scores (Raw Logits):
 - on: -0.2298
 - off: 0.0129
 - stop: 0.1427
 - unknown: 0.0743
